# Visual Novel — Render video bằng Google Colab

Notebook này nhận gói ZIP xuất từ nút **Chuẩn bị gói render Colab** trong `phong_video.html`. Colab tự tải tài nguyên gốc từ GitHub, dựng hình từ timeline và ghép WAV/BGM/SFX thành MP4. Không cần xuất hoặc tải hàng nghìn ảnh khung hình từ trình duyệt.

**Lưu ý:** Colab miễn phí có giới hạn thời gian/tài nguyên; GPU không được đảm bảo. Video dài hoặc 60 FPS có thể render lâu. Engine hiện là bản đầu, chưa bảo đảm giống pixel-perfect với trình duyệt; hãy thử một đoạn ngắn trước.

In [ ]:
!pip -q install av opencv-python-headless pillow numpy

## 1. Kết nối Google Drive và tải gói ZIP lên

ZIP chỉ chứa timeline, WAV thoại và tham chiếu tài nguyên. Các ảnh nền, video nhân vật và âm thanh thư viện được Colab tải trực tiếp từ repository.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import re

drive.mount('/content/drive')
uploaded = files.upload()  # Chọn file *_colab_render_package.zip vừa xuất từ phòng video
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise RuntimeError('Bạn chưa tải lên file ZIP gói render Colab.')
ZIP_PATH = Path('/content') / zip_names[0]
print('Đã nhận gói:', ZIP_PATH.name, '-', round(ZIP_PATH.stat().st_size/1024/1024, 1), 'MB')

## 2. Tải engine và bắt đầu render

Có thể đổi `OUTPUT_DIR` hoặc `OUTPUT_NAME`. File MP4 sẽ được lưu trong Google Drive để tránh mất khi phiên Colab kết thúc.

In [ ]:
import requests, subprocess, sys

ENGINE_URL = 'https://raw.githubusercontent.com/nhothicam01-png/visual-novel-game/main/colab-render/renderer.py'
engine_response = requests.get(ENGINE_URL, timeout=60)
engine_response.raise_for_status()
ENGINE_PATH = Path('/content/renderer.py')
ENGINE_PATH.write_text(engine_response.text, encoding='utf-8')

OUTPUT_DIR = Path('/content/drive/MyDrive/VisualNovelRenders')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_NAME = re.sub(r'[^A-Za-z0-9_-]+', '_', ZIP_PATH.stem.replace('_colab_render_package','')) + '_colab.mp4'
OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME

command = [sys.executable, str(ENGINE_PATH), '--zip', str(ZIP_PATH), '--output', str(OUTPUT_PATH)]
print('Lệnh render:', ' '.join(command))
subprocess.run(command, check=True)
print('Video đã lưu:', OUTPUT_PATH)

## 3. Tải MP4 về thiết bị (tùy chọn)

Nếu video lớn, bạn có thể mở Google Drive để lấy file thay vì tải trực tiếp trong Colab.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))